<a href="https://colab.research.google.com/github/elhaithamy/budget-expenses-manager/blob/main/Sort_My_Money.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
import gspread
from google.oauth2.service_account import Credentials

# ====== PAGE CONFIG ======
st.set_page_config(
    page_title="Family Budget Dashboard 2026",
    page_icon="💰",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ====== GOOGLE SHEETS CONNECTION ======
@st.cache_data(ttl=600)  # Cache for 10 minutes
def load_google_sheet(sheet_url):
    """Load data from Google Sheets"""
    try:
        # For public sheets (simplest for personal use)
        sheet_id = sheet_url.split('/d/')[1].split('/')[0]
        csv_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"
        df = pd.read_csv(csv_url)
        return df
    except Exception as e:
        st.error(f"Error loading sheet: {e}")
        return None

# ====== INITIALIZE SESSION STATE ======
if 'month_state' not in st.session_state:
    # 12 months of data
    st.session_state.month_state = [
        {
            'income': {'salary_planned': 15000, 'salary_actual': 15000,
                      'bonus_planned': 0, 'bonus_actual': 0, 'notes': ''},
            'fixed': {'housing_p': 5250, 'housing_a': 5250,
                     'insurance_p': 600, 'insurance_a': 600,
                     'education_p': 1500, 'education_a': 1500,
                     'transport_p': 2250, 'transport_a': 2250,
                     'comm_p': 300, 'comm_a': 300, 'notes': ''},
            'variable': {'food_p': 3000, 'food_a': 3000,
                        'util_p': 1500, 'util_a': 1500,
                        'ent_p': 450, 'ent_a': 450,
                        'seasonal_p': 1000, 'seasonal_a': 1000, 'notes': ''},
            'investments': {'emergency_p': 1200, 'emergency_a': 1200,
                          'stock_p': 800, 'stock_a': 800,
                          're_p': 500, 're_a': 500,
                          'edu_p': 400, 'edu_a': 400,
                          'ret_p': 600, 'ret_a': 600,
                          'metal_grams_p': 0, 'metal_grams_a': 0, 'notes': ''}
        } for _ in range(12)
    ]

if 'fixed_settings' not in st.session_state:
    st.session_state.fixed_settings = {
        'base_salary': 15000,
        'target_savings_pct': 40,
        'emergency_target': 90000,
        'portfolio_target': 160000,
        'gold_price_per_gram': 0,
        'silver_price_per_gram': 0
    }

# ====== SIDEBAR ======
with st.sidebar:
    st.title("💰 Budget 2026")

    # Today's date
    today = datetime.now()
    st.info(f"📅 Today: {today.strftime('%d %b %Y')}")

    # Month selector
    months = ['January', 'February', 'March', 'April', 'May', 'June',
              'July', 'August', 'September', 'October', 'November', 'December']
    active_month = st.selectbox(
        "Active Month",
        range(12),
        format_func=lambda x: months[x],
        index=today.month - 1
    )

    st.divider()

    # Google Sheet URL
    with st.expander("⚙️ Data Source"):
        sheet_url = st.text_input(
            "Google Sheets URL",
            placeholder="Paste your share link here",
            help="Make sure the sheet is shared (Anyone with link can view)"
        )
        if sheet_url and st.button("Load from Sheet"):
            df = load_google_sheet(sheet_url)
            if df is not None:
                st.success("✅ Sheet loaded!")
                st.dataframe(df.head())

    st.divider()

    # Fixed Settings
    with st.expander("🎯 Fixed Settings"):
        st.session_state.fixed_settings['base_salary'] = st.number_input(
            "Base Salary", value=15000, step=100
        )
        st.session_state.fixed_settings['target_savings_pct'] = st.number_input(
            "Target Savings % of Base", value=40, step=1
        )
        st.session_state.fixed_settings['emergency_target'] = st.number_input(
            "Emergency Fund Target", value=90000, step=1000
        )
        st.session_state.fixed_settings['gold_price_per_gram'] = st.number_input(
            "Gold Price per Gram", value=0.0, step=0.01, format="%.2f"
        )

# ====== MAIN CONTENT ======
tab1, tab2, tab3, tab4, tab5 = st.tabs([
    "📊 Dashboard", "📥 Inputs", "📆 Daily & Weekly", "📈 Investments", "📉 Charts"
])

# ====== TAB 1: DASHBOARD ======
with tab1:
    st.header(f"📊 {months[active_month]} Overview")

    # Get current month data
    m = st.session_state.month_state[active_month]
    fs = st.session_state.fixed_settings

    # Calculate totals
    income_planned = m['income']['salary_planned'] + m['income']['bonus_planned']
    income_actual = m['income']['salary_actual'] + m['income']['bonus_actual']

    fixed_planned = sum([m['fixed'][k] for k in m['fixed'] if k.endswith('_p')])
    fixed_actual = sum([m['fixed'][k] for k in m['fixed'] if k.endswith('_a')])

    var_planned = sum([m['variable'][k] for k in m['variable'] if k.endswith('_p')])
    var_actual = sum([m['variable'][k] for k in m['variable'] if k.endswith('_a')])

    inv_planned = sum([m['investments'][k] for k in m['investments'] if k.endswith('_p') and k != 'metal_grams_p'])
    inv_actual = sum([m['investments'][k] for k in m['investments'] if k.endswith('_a') and k != 'metal_grams_a'])

    # Add metals value
    metal_value_planned = m['investments']['metal_grams_p'] * fs['gold_price_per_gram']
    metal_value_actual = m['investments']['metal_grams_a'] * fs['gold_price_per_gram']
    inv_planned += metal_value_planned
    inv_actual += metal_value_actual

    total_expense_actual = fixed_actual + var_actual

    # KPIs
    col1, col2, col3, col4 = st.columns(4)

    with col1:
        st.metric("💰 Income", f"{income_actual:,.0f}",
                 f"{income_actual - income_planned:+,.0f}")

    with col2:
        st.metric("💸 Expenses", f"{total_expense_actual:,.0f}",
                 f"{total_expense_actual - (fixed_planned + var_planned):+,.0f}")

    with col3:
        st.metric("📈 Investments", f"{inv_actual:,.0f}",
                 f"{inv_actual - inv_planned:+,.0f}")

    with col4:
        target_savings = fs['base_salary'] * fs['target_savings_pct'] / 100
        savings_pct = (inv_actual / fs['base_salary']) * 100 if fs['base_salary'] > 0 else 0
        st.metric("🎯 Savings vs Target", f"{savings_pct:.1f}%",
                 f"{savings_pct - fs['target_savings_pct']:.1f}%")

    st.divider()

    # Planned vs Actual Tables
    col1, col2 = st.columns(2)

    with col1:
        st.subheader("💰 Income")
        income_df = pd.DataFrame({
            'Category': ['Salary', 'Bonus'],
            'Planned': [m['income']['salary_planned'], m['income']['bonus_planned']],
            'Actual': [m['income']['salary_actual'], m['income']['bonus_actual']],
        })
        income_df['Variance'] = income_df['Actual'] - income_df['Planned']
        st.dataframe(income_df, use_container_width=True)

    with col2:
        st.subheader("💸 Expenses Summary")
        exp_df = pd.DataFrame({
            'Category': ['Fixed Bills', 'Variable Spending'],
            'Planned': [fixed_planned, var_planned],
            'Actual': [fixed_actual, var_actual],
        })
        exp_df['Variance'] = exp_df['Actual'] - exp_df['Planned']
        st.dataframe(exp_df, use_container_width=True)

# ====== TAB 2: INPUTS ======
with tab2:
    st.header(f"📥 {months[active_month]} Inputs")

    m = st.session_state.month_state[active_month]

    with st.expander("💰 Income", expanded=True):
        col1, col2 = st.columns(2)
        with col1:
            m['income']['salary_planned'] = st.number_input(
                "Salary - Planned", value=m['income']['salary_planned'], step=100
            )
        with col2:
            m['income']['salary_actual'] = st.number_input(
                "Salary - Actual", value=m['income']['salary_actual'], step=100
            )

        col1, col2 = st.columns(2)
        with col1:
            m['income']['bonus_planned'] = st.number_input(
                "Bonus - Planned", value=m['income']['bonus_planned'], step=100
            )
        with col2:
            m['income']['bonus_actual'] = st.number_input(
                "Bonus - Actual", value=m['income']['bonus_actual'], step=100
            )

        m['income']['notes'] = st.text_area("Income Notes", value=m['income']['notes'])

    with st.expander("💳 Fixed Bills"):
        col1, col2 = st.columns(2)
        with col1:
            m['fixed']['housing_p'] = st.number_input("Housing - Planned", value=m['fixed']['housing_p'])
        with col2:
            m['fixed']['housing_a'] = st.number_input("Housing - Actual", value=m['fixed']['housing_a'])

        # Add more fixed bills similarly...

    with st.expander("🍽️ Variable Spending"):
        col1, col2 = st.columns(2)
        with col1:
            m['variable']['food_p'] = st.number_input("Food - Planned", value=m['variable']['food_p'])
        with col2:
            m['variable']['food_a'] = st.number_input("Food - Actual", value=m['variable']['food_a'])

        # Add more variable categories...

    with st.expander("📈 Investments"):
        col1, col2 = st.columns(2)
        with col1:
            m['investments']['emergency_p'] = st.number_input("Emergency - Planned", value=m['investments']['emergency_p'])
        with col2:
            m['investments']['emergency_a'] = st.number_input("Emergency - Actual", value=m['investments']['emergency_a'])

        # Add more investment types...

        st.subheader("🥇 Gold/Silver")
        col1, col2 = st.columns(2)
        with col1:
            m['investments']['metal_grams_p'] = st.number_input("Grams - Planned", value=m['investments']['metal_grams_p'])
        with col2:
            m['investments']['metal_grams_a'] = st.number_input("Grams - Actual", value=m['investments']['metal_grams_a'])

        if st.session_state.fixed_settings['gold_price_per_gram'] > 0:
            metal_value = m['investments']['metal_grams_a'] * st.session_state.fixed_settings['gold_price_per_gram']
            st.info(f"💰 Current value: {metal_value:,.2f}")

    if st.button("✅ Save Changes", type="primary"):
        st.success("✅ Changes saved!")
        st.rerun()

# ====== TAB 3: DAILY & WEEKLY ======
with tab3:
    st.header(f"📆 Daily & Weekly - {months[active_month]}")
    st.info("Daily tracking coming in next update - foundation is ready!")

# ====== TAB 4: INVESTMENTS ======
with tab4:
    st.header("📈 Investment Portfolio")

    # Portfolio summary across all months
    portfolio_data = []
    for i, month_name in enumerate(months):
        m = st.session_state.month_state[i]
        total_inv = sum([m['investments'][k] for k in m['investments'] if k.endswith('_a') and k != 'metal_grams_a'])
        portfolio_data.append({'Month': month_name, 'Investment': total_inv})

    df = pd.DataFrame(portfolio_data)
    df['Cumulative'] = df['Investment'].cumsum()

    fig = px.line(df, x='Month', y='Cumulative', title='Cumulative Investments 2026',
                  markers=True, labels={'Cumulative': 'Total Invested'})
    st.plotly_chart(fig, use_container_width=True)

# ====== TAB 5: CHARTS ======
with tab5:
    st.header("📉 Financial Charts")

    # Income vs Expenses by Month
    chart_data = []
    for i, month_name in enumerate(months):
        m = st.session_state.month_state[i]
        income = m['income']['salary_actual'] + m['income']['bonus_actual']
        expenses = sum([m['fixed'][k] for k in m['fixed'] if k.endswith('_a')]) + \
                  sum([m['variable'][k] for k in m['variable'] if k.endswith('_a')])
        chart_data.append({
            'Month': month_name,
            'Income': income,
            'Expenses': expenses
        })

    df_chart = pd.DataFrame(chart_data)

    fig = go.Figure()
    fig.add_trace(go.Bar(x=df_chart['Month'], y=df_chart['Income'], name='Income', marker_color='green'))
    fig.add_trace(go.Bar(x=df_chart['Month'], y=df_chart['Expenses'], name='Expenses', marker_color='red'))
    fig.update_layout(title='Income vs Expenses 2026', barmode='group')

    st.plotly_chart(fig, use_container_width=True)

# ====== FOOTER ======
st.divider()
st.caption("💰 Family Budget Dashboard 2026 | Built with Streamlit")
share.streamlit.io

ModuleNotFoundError: No module named 'streamlit'

In [2]:
pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 78.1 MB/s eta 0:00:00
